# Optimisation du CNN avec CBAM — v2 (connexion résiduelle) — apprentissage et validation

Ce notebook est autonome : ouvrir avec le noyau **memoire_cnn**, puis exécuter les cellules dans l'ordre.
Il entraîne six modèles depuis zéro, chacun avec le même état initial et le même ordre initial des lots.
Il ne lit aucun fichier de test. Les expériences précédentes (v1, dossier `cnn_cbam_grille_v1`) sont conservées et ne sont pas modifiées : ce notebook écrit dans un nouveau dossier `cnn_cbam_grille_v2_residuel`.

**Ce qui change par rapport à la v1, et pourquoi** : dans la v1, le module CBAM se terminait par deux `sigmoid()` (attention canal puis spatiale). À l'initialisation, avec des poids proches de zéro, `sigmoid(≈0) ≈ 0.5`, donc le signal transmis au classificateur était atténué d'environ 0,5 × 0,5 ≈ 0,25 dès le départ — avant même que l'entraînement ne commence. Avec seulement 20 époques et une patience de 4, le modèle devait à la fois apprendre la tâche **et** compenser ce handicap de départ, dans le même budget que le CNN classique qui n'a pas ce problème. C'est la cause la plus probable d'un F1 macro plus faible pour CBAM.

**Correction appliquée (v2)** : une connexion résiduelle est ajoutée autour du bloc CBAM (`x + attention(x)` au lieu de `attention(x)` seul). Ainsi, à l'initialisation, le module se comporte comme une quasi-identité et n'atténue plus le signal ; le gradient apprend ensuite progressivement *où* renforcer ou atténuer les features, plutôt que de devoir d'abord réparer un signal divisé par quatre.

**Ce qui reste strictement identique au notebook 05 (CNN classique) et à la v1 de ce notebook**, pour que la comparaison reste valide et n'attribue l'effet observé qu'à ce seul changement :
- la grille d'hyperparamètres (LR ∈ {0.001, 0.0005, 0.0001} ; Dropout ∈ {0.3, 0.5}) ;
- le budget (maximum 20 époques, patience 4 sur le F1 macro de validation, critère d'arrêt, départage par ordre de grille) ;
- la graine (42), la taille de lot (16), l'optimiseur (Adam, weight_decay 0.0) ;
- le prétraitement, la répartition train/val, les poids de classes ;
- l'initialisation partagée des couches communes (le CBAM hérite de `CNNClassique`, donc bloc1-3 et classificateur démarrent avec exactement les mêmes poids que le CNN classique et que la v1 CBAM) ;
- le ratio de réduction du CBAM (16) et son unique emplacement (après le bloc 3).

Un seul changement architectural est testé ici : l'ajout de la connexion résiduelle. C'est volontaire — isoler ce changement permet d'attribuer sans ambiguïté un éventuel gain de F1 macro à cette correction précise, plutôt qu'à un ensemble de modifications combinées. Si le F1 macro de CBAM reste inférieur à celui du CNN classique après cette correction, d'autres leviers pourront être testés ensuite, un par un : ratio de réduction (8 au lieu de 16), placement du CBAM (après le bloc 2 en plus du bloc 3), patience allongée (6-8), ou une légère régularisation L2 (weight_decay). Ces leviers ne sont pas mélangés ici pour garder une attribution causale claire.

La recherche sur une seule graine ne démontre pas encore la robustesse. Les configurations retenues devront ensuite être examinées sur plusieurs graines (notebook 07) et selon un protocole de fiabilité défini à l'avance.

**Reprise** : après interruption, redémarrer le noyau et réexécuter ce notebook. La dernière époque complète sera rechargée ; une époque interrompue sera recommencée. Ne pas exécuter deux copies simultanément dans le même dossier. Ne pas changer le protocole au milieu de la recherche.


In [ ]:
from pathlib import Path
import json, hashlib, time, shutil, sys
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn import __version__ as version_sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm.auto import tqdm
from IPython.display import display

PROJET = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention")
IMAGES = PROJET / "archive" / "chest_xray"
REPARTITION = PROJET / "repartitions" / "repartition_20260917_102959_713272"
SORTIE = PROJET / "optimisation" / "cnn_cbam_grille_v2_residuel"
GRAINE, LOT, MAX_EPOQUES, PATIENCE = 42, 16, 20, 4
APPAREIL = torch.device("cpu")
GRILLE = [(lr, dropout) for lr in [0.001, 0.0005, 0.0001] for dropout in [0.3, 0.5]]
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}
torch.use_deterministic_algorithms(True)
print("Matériel :", APPAREIL, "| PyTorch :", torch.__version__)
display(pd.DataFrame(GRILLE, columns=["taux_apprentissage", "dropout"]))


## Vérification des données
Les poids des classes sont calculés uniquement sur l'apprentissage. Prétraitement inchangé : niveaux de gris, padding noir centré, interpolation bilinéaire, 224 × 224, division par 255, aucune augmentation.


In [ ]:
train_df = pd.read_csv(REPARTITION / "train.csv")
val_df = pd.read_csv(REPARTITION / "val.csv")
for nom, df, attendus in [("train", train_df, {"NORMAL": 1079, "PNEUMONIA": 2721}), ("val", val_df, {"NORMAL": 269, "PNEUMONIA": 684})]:
    assert df["classe"].value_counts().to_dict() == attendus, f"Effectifs inattendus : {nom}"
    assert df["chemin_relatif"].is_unique
    assert not df[["groupe", "empreinte", "chemin_relatif"]].isna().any().any()
    manquants = [p for p in df["chemin_relatif"] if not (IMAGES / p).is_file()]
    if manquants:
        raise FileNotFoundError(f"{nom} : {len(manquants)} images absentes, exemple : {manquants[0]}")
for colonne in ["groupe", "empreinte", "chemin_relatif"]:
    assert set(train_df[colonne]).isdisjoint(val_df[colonne]), f"Chevauchement : {colonne}"

class Radiographies(Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        ligne = self.df.iloc[i]
        with Image.open(IMAGES / ligne["chemin_relatif"]) as image:
            image = ImageOps.pad(image.convert("L"), (224, 224), method=Image.Resampling.BILINEAR, color=0, centering=(0.5, 0.5))
            x = torch.from_numpy(np.array(image, dtype=np.float32) / 255.0).unsqueeze(0)
        return x, CLASSES[ligne["classe"]]

TRAIN, VAL = Radiographies(train_df), Radiographies(val_df)
effectifs = train_df["classe"].value_counts()
POIDS = torch.tensor([len(TRAIN)/(2*effectifs[c]) for c in ["NORMAL", "PNEUMONIA"]], dtype=torch.float32)
print("Apprentissage :", len(TRAIN), "| Validation :", len(VAL))
print("Poids NORMAL / PNEUMONIA :", POIDS.tolist())
print("Vérifications réussies. Aucun ensemble de test chargé.")


## Architecture et contrôle d'un lot
Le CNN de base est conservé à l'identique, avec un module CBAM après le troisième bloc (64 canaux, réduction 16, noyau spatial 7 — inchangé par rapport à la v1). Le Dropout est configurable. Total : **89 636 paramètres, dont 610 pour CBAM — identique à la v1**, car la connexion résiduelle n'ajoute aucun paramètre (c'est une simple addition, `x + attention(x)`) : seul le comportement du module à l'initialisation change, pas sa capacité. Les couches communes (bloc1-3, classificateur) sont initialisées avant l'ajout de CBAM, donc avec exactement les mêmes poids que le CNN classique du notebook 05.


In [ ]:
class CNNClassique(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc3 = nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.reduction = nn.AdaptiveAvgPool2d((4,4))
        self.classificateur = nn.Sequential(nn.Flatten(),nn.Linear(64*4*4,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,2))
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(x))

class AttentionCanaux(nn.Module):
    def __init__(self, canaux, reduction=16):
        super().__init__()
        dim=max(1,canaux//reduction)
        self.reseau_partage=nn.Sequential(
            nn.Conv2d(canaux,dim,1,bias=False),nn.ReLU(),
            nn.Conv2d(dim,canaux,1,bias=False))
    def forward(self,x):
        moyenne=x.mean(dim=(2,3),keepdim=True)
        maximum=x.amax(dim=(2,3),keepdim=True)
        return x*torch.sigmoid(self.reseau_partage(moyenne)+self.reseau_partage(maximum))

class AttentionSpatiale(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution=nn.Conv2d(2,1,7,padding=3,bias=False)
    def forward(self,x):
        resume=torch.cat([x.mean(dim=1,keepdim=True),x.amax(dim=1,keepdim=True)],dim=1)
        return x*torch.sigmoid(self.convolution(resume))

class CBAM(nn.Module):
    """v2 : connexion résiduelle ajoutée autour de l'attention canal + spatiale.
    À l'initialisation, les deux sigmoïdes valent environ 0.5, donc `attention_spatiale(attention_canaux(x))`
    seul atténuerait x d'un facteur ~0.25 avant même le début de l'entraînement (voir cellule 0).
    En renvoyant x + attention(x) plutôt que attention(x) seul, le module démarre en quasi-identité
    (le terme résiduel x domine, le terme d'attention est une perturbation additive au départ)
    et n'ajoute aucun paramètre : c'est un changement de comportement à l'initialisation, pas de capacité."""
    def __init__(self,canaux=64,reduction=16):
        super().__init__()
        self.attention_canaux=AttentionCanaux(canaux,reduction)
        self.attention_spatiale=AttentionSpatiale()
    def forward(self,x):
        attenue = self.attention_spatiale(self.attention_canaux(x))
        return x + attenue

class CNNAvecCBAM(CNNClassique):
    def __init__(self,dropout=0.3):
        super().__init__(dropout)
        self.cbam=CBAM(64,16)
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(self.cbam(x)))

with torch.random.fork_rng(devices=[]):
    modele_verification = CNNAvecCBAM().eval()
    x, y = next(iter(DataLoader(TRAIN, batch_size=LOT, shuffle=False, num_workers=0)))
    with torch.no_grad(): scores = modele_verification(x)
    assert scores.shape == (LOT, 2) and torch.isfinite(scores).all()
    assert sum(p.numel() for p in modele_verification.parameters()) == 89636
del modele_verification
print("Architecture vérifiée : 89 636 paramètres (identique à la v1), deux scores par image.")
print("CBAM v2 : connexion résiduelle active (x + attention(x)), aucun paramètre supplémentaire.")


## Fonctions d'entraînement et de sauvegarde
La perte affichée est une moyenne pondérée correcte sur tous les lots. Les mesures d'apprentissage sont relevées pendant les mises à jour avec Dropout, celles de validation après l'époque en mode évaluation.
Chaque époque complète est écrite dans un fichier distinct. Les sauvegardes contiennent aussi l'optimiseur et les états aléatoires nécessaires à la reprise dans le même environnement. Source : [documentation PyTorch](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).


In [ ]:
def ecrire_json(obj, chemin):
    tmp = chemin.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(chemin)

def sauver_torch(obj, chemin):
    tmp = chemin.with_suffix(".tmp")
    torch.save(obj, tmp)
    tmp.replace(chemin)

PROTOCOLE = {
    "architecture": "CNNAvecCBAM_89636_apres_bloc3_r16_k7_residuel_v2", "graine": GRAINE,
    "lot": LOT, "max_epoques": MAX_EPOQUES, "patience": PATIENCE,
    "critere": "val_f1_macro", "amelioration": "strictement superieure",
    "grille": [list(x) for x in GRILLE], "optimiseur": "Adam", "weight_decay": 0.0,
    "pretraitement": "L; ImageOps.pad 224 bilineaire noir centre; /255; aucune augmentation",
    "classes": CLASSES, "poids_classes": POIDS.tolist(),
    "sha256_csv": {s: hashlib.sha256((REPARTITION / f"{s}.csv").read_bytes()).hexdigest() for s in ["train", "val"]},
    "versions": {"python": sys.version, "torch": str(torch.__version__), "numpy": np.__version__, "pandas": pd.__version__, "pillow": PIL.__version__, "sklearn": version_sklearn},
    "threads_torch": torch.get_num_threads(),
    "correction_v2": "Ajout d'une connexion residuelle autour du CBAM (x + attention(x) au lieu de attention(x) seul). "
                      "Motif : les deux sigmoides du CBAM valent ~0.5 a l'initialisation, ce qui attenuait le signal "
                      "d'un facteur ~0.25 avant meme le debut de l'entrainement dans la v1 (dossier cnn_cbam_grille_v1). "
                      "Aucun autre element du protocole n'a change par rapport au notebook 05 (CNN classique) ni a la v1 de ce notebook : "
                      "meme grille, meme budget (20 epoques, patience 4), meme graine, meme ratio de reduction (16), meme emplacement (apres bloc 3).",
    "limite": "Recherche exploratoire apres consultation anterieure du test. Robustesse multi-graines non encore mesuree. "
              "Un seul changement architectural teste ici (residuel) pour permettre une attribution causale claire ; "
              "si le F1 macro reste inferieur au CNN classique apres cette correction, d'autres leviers (ratio de reduction, "
              "placement multi-blocs, patience allongee, weight_decay) restent a tester un par un, pas combines."
}
SORTIE.mkdir(parents=True, exist_ok=True)
fichier_protocole = SORTIE / "protocole.json"
if fichier_protocole.exists():
    if json.loads(fichier_protocole.read_text(encoding="utf-8")) != PROTOCOLE:
        raise RuntimeError("Protocole ou environnement différent : reprendre l'environnement initial ou choisir un nouveau dossier SORTIE.")
else:
    ecrire_json(PROTOCOLE, fichier_protocole)

def une_epoque(modele, chargeur, critere, optimiseur=None):
    apprentissage = optimiseur is not None
    modele.train(apprentissage)
    pertes, poids_total = 0.0, 0.0
    reels, predits = [], []
    for images, etiquettes in tqdm(chargeur, desc="Apprentissage" if apprentissage else "Validation", leave=False):
        images, etiquettes = images.to(APPAREIL), etiquettes.to(APPAREIL)
        if apprentissage: optimiseur.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(apprentissage):
            scores = modele(images)
            perte = critere(scores, etiquettes)
            if not torch.isfinite(perte).item(): raise RuntimeError("Perte non finie : arrêter et examiner cette configuration.")
            if apprentissage:
                perte.backward()
                optimiseur.step()
        poids = critere.weight[etiquettes].sum().item()
        pertes += perte.item()*poids
        poids_total += poids
        reels.extend(etiquettes.cpu().tolist())
        predits.extend(scores.detach().argmax(1).cpu().tolist())
    cm = confusion_matrix(reels, predits, labels=[0,1])
    tn,fp,fn,tp = cm.ravel()
    mesures = {
        "perte": float(pertes/poids_total), "exactitude": float(accuracy_score(reels,predits)),
        "precision_pneumonie": float(precision_score(reels,predits,zero_division=0)),
        "rappel_pneumonie": float(recall_score(reels,predits,zero_division=0)),
        "specificite_NORMAL": float(tn/(tn+fp)),
        "f1_pneumonie": float(f1_score(reels,predits,zero_division=0)),
        "f1_macro": float(f1_score(reels,predits,labels=[0,1],average="macro",zero_division=0))
    }
    return mesures, cm.tolist()

def lancer_configuration(indice, lr, dropout):
    dossier = SORTIE / f"configuration_{indice:02d}"
    dossier.mkdir(exist_ok=True)
    configuration = {"indice": indice, "lr": lr, "dropout": dropout}
    torch.manual_seed(GRAINE)
    modele = CNNAvecCBAM(dropout).to(APPAREIL)
    optimiseur = torch.optim.Adam(modele.parameters(), lr=lr)
    critere = nn.CrossEntropyLoss(weight=POIDS.to(APPAREIL))
    generateur = torch.Generator().manual_seed(GRAINE)
    chargeur_train = DataLoader(TRAIN,batch_size=LOT,shuffle=True,generator=generateur,num_workers=0)
    chargeur_val = DataLoader(VAL,batch_size=LOT,shuffle=False,num_workers=0)
    # Graine d'entraînement commune, indépendante du nombre de paramètres initialisés.
    torch.manual_seed(GRAINE)
    historique = []
    def sauvegarder(epoque, matrice=None):
        sauver_torch({"epoque":epoque,"modele":modele.state_dict(),"optimiseur":optimiseur.state_dict(),
                      "historique":historique,"configuration":configuration,"protocole":PROTOCOLE,
                      "etat_torch":torch.get_rng_state(),"etat_generateur":generateur.get_state(),
                      "matrice_validation":matrice}, dossier/f"epoque_{epoque:03d}.pt")
    disponibles = sorted(dossier.glob("epoque_[0-9][0-9][0-9].pt"))
    if disponibles:
        etat = torch.load(disponibles[-1], map_location="cpu", weights_only=True)
        assert etat["configuration"] == configuration and etat["protocole"] == PROTOCOLE
        modele.load_state_dict(etat["modele"],strict=True)
        optimiseur.load_state_dict(etat["optimiseur"])
        historique = etat["historique"]
        assert etat["epoque"] == len(historique)
        torch.set_rng_state(etat["etat_torch"])
        generateur.set_state(etat["etat_generateur"])
        print("Reprise après l'époque", etat["epoque"])
    else:
        sauvegarder(0)
    while len(historique) < MAX_EPOQUES:
        if historique:
            meilleur = max(historique,key=lambda r:r["val_f1_macro"])
            if len(historique)-meilleur["epoque"] >= PATIENCE: break
        epoque = len(historique)+1
        debut = time.perf_counter()
        tr,_ = une_epoque(modele,chargeur_train,critere,optimiseur)
        va,cm = une_epoque(modele,chargeur_val,critere)
        ligne = {"epoque":epoque,"duree_secondes":float(time.perf_counter()-debut)}
        ligne.update({f"train_{k}":v for k,v in tr.items()})
        ligne.update({f"val_{k}":v for k,v in va.items()})
        historique.append(ligne)
        sauvegarder(epoque,cm)
        pd.DataFrame(historique).to_csv(dossier/"historique.csv",index=False)
        print(f"Époque {epoque:02d} | F1 macro val {va['f1_macro']:.4f} | rappel {va['rappel_pneumonie']:.4f} | spécificité {va['specificite_NORMAL']:.4f}")
    meilleur = max(historique,key=lambda r:r["val_f1_macro"])
    source = dossier/f"epoque_{meilleur['epoque']:03d}.pt"
    tmp = dossier/"meilleur.tmp"
    shutil.copy2(source,tmp)
    tmp.replace(dossier/"meilleur.pt")
    pd.DataFrame(historique).to_csv(dossier/"historique.csv",index=False)
    resume = {**configuration,"epoque_retenue":meilleur["epoque"],"epoque_finale":len(historique),
              "arret": "patience" if len(historique)-meilleur["epoque"]>=PATIENCE else "maximum_epoques",
              "duree_minutes":sum(r["duree_secondes"] for r in historique)/60,
              "sauvegarde":str(dossier/"meilleur.pt"),
              **{k:v for k,v in meilleur.items() if k.startswith("val_")}}
    ecrire_json(resume,dossier/"resume.json")
    return resume

print("Fonctions et protocole prêts. Aucun entraînement lancé jusqu'ici.")


## Lancement des six configurations
**Cette cellule lance les entraînements, successivement sur CPU.** Maximum : 120 époques au total, soit environ 7 à 8 heures si chaque époque dure 3,5 à 4 minutes ; l'arrêt anticipé peut réduire cette durée. Laisser le PC alimenté et éviter la mise en veille.

Pour faire une pause, interrompre la cellule. À la reprise, redémarrer le noyau puis exécuter toutes les cellules dans l'ordre. Les configurations déjà terminées seront reconnues à partir de leur historique et ne seront pas réentraînées.

**Astuce avant de lancer les 6 configurations complètes** : envisage d'abord un essai rapide sur 1 à 2 configurations (par exemple en réduisant temporairement `GRILLE` à `GRILLE[:2]` dans une cellule de test séparée, sans modifier cette cellule) pour vérifier que le F1 macro progresse normalement avec la connexion résiduelle, avant d'investir les 7-8 heures complètes.


In [ ]:
resumes = []
for indice, (lr, dropout) in enumerate(GRILLE, start=1):
    print(f"\n===== Configuration {indice}/6 : LR={lr}, Dropout={dropout} =====")
    resume = lancer_configuration(indice, lr, dropout)
    resumes.append(resume)
    bilan = pd.DataFrame(resumes)
    bilan.to_csv(SORTIE / "comparaison_configurations.csv", index=False, encoding="utf-8-sig")
    display(bilan[["indice","lr","dropout","epoque_retenue","epoque_finale","val_f1_macro","val_rappel_pneumonie","val_specificite_NORMAL"]].round(4))

retenue = max(resumes, key=lambda r:r["val_f1_macro"])
ecrire_json(retenue, SORTIE / "configuration_retenue.json")
print("\nRecherche terminée. Configuration retenue :", retenue["indice"])
print("F1 macro de validation :", retenue["val_f1_macro"])
print("Tableau à transmettre :", SORTIE / "comparaison_configurations.csv")
print("Aucun test exécuté. La stabilité sur plusieurs graines reste à examiner.")
